# Python CDC Pipeline Tests (python_cdc.ipynb)

Unit tests for `transformations/python_cdc.ipynb` using **dbx_test** framework.

Tests the DLT pipeline logic including:
- Dynamic pipeline creation from folder listing
- Data quality expectations (@dlt.expect_or_drop)
- APPLY CHANGES SCD Type 1 and SCD Type 2

**Run via CLI:** `dbx_test run --config config/test_config.yml --output-format html,json,junit --output-dir .dbx-test-results`

In [ ]:
%run ./conftest

In [ ]:
from dbx_test import NotebookTestFixture, run_notebook_tests
import pytest
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, row_number, expr, lit
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType
from datetime import datetime, timedelta
from typing import List, Dict, Any

In [ ]:
# Helper functions extracted from python_cdc.ipynb logic (testable transformations)
# DLT decorators cannot be tested directly - we test the equivalent PySpark logic

def apply_dlt_expectations(df: DataFrame, no_rescued_data=True, valid_id=True, valid_operation=True) -> DataFrame:
    """Simulates @dlt.expect_or_drop for no_rescued_data, valid_id, valid_operation"""
    valid_ops = ['APPEND', 'DELETE', 'UPDATE']
    result = df
    if no_rescued_data and "_rescued_data" in df.columns:
        result = result.filter(col("_rescued_data").isNull())
    if valid_id:
        result = result.filter(col("id").isNotNull())
    if valid_operation:
        result = result.filter(col("operation").isin(valid_ops))
    return result

def deduplicate_by_sequence(df: DataFrame, keys=None, sequence_col="operation_date") -> DataFrame:
    """Simulates SEQUENCE BY in dlt.apply_changes"""
    keys = keys or ["id"]
    w = Window.partitionBy(*keys).orderBy(col(sequence_col).desc())
    return df.withColumn("_rank", row_number().over(w)).filter(col("_rank") == 1).drop("_rank")

def apply_changes_scd1(source_df: DataFrame, target_df: DataFrame, keys=None, delete_expr="operation = 'DELETE'", except_cols=None) -> DataFrame:
    """Simulates dlt.apply_changes() default SCD1 behavior"""
    keys = keys or ["id"]
    except_cols = except_cols or ["operation", "operation_date", "_rescued_data"]
    w = Window.partitionBy(*keys).orderBy(col("operation_date").desc())
    source_deduped = source_df.withColumn("_r", row_number().over(w)).filter(col("_r") == 1).drop("_r")
    deletes = source_deduped.filter(expr(delete_expr))
    delete_ids = set(row["id"] for row in deletes.select(keys).collect())
    upserts = source_deduped.filter(~expr(delete_expr))
    upsert_ids = set(row["id"] for row in upserts.select(keys).collect())
    all_affected = delete_ids | upsert_ids
    target_filtered = target_df.filter(~col("id").isin(list(all_affected)))
    existing_except = [c for c in except_cols if c in upserts.columns]
    upserts_prepared = upserts.drop(*existing_except)
    result_cols = target_filtered.columns
    upserts_final = upserts_prepared.select(*[col(c) if c in upserts_prepared.columns else lit(None).alias(c) for c in result_cols])
    return target_filtered.union(upserts_final)

def get_table_names_from_folders(folder_listing: List[Any]) -> List[str]:
    """Extract table names from dbutils.fs.ls() - folder.name[:-1]"""
    return [f.name[:-1] for f in folder_listing]

def create_raw_cdc_path(table_name: str, catalog: str, schema: str) -> str:
    """Path for raw CDC: /Volumes/{catalog}/{schema}/raw_data/{table}"""
    return f"/Volumes/{catalog}/{schema}/raw_data/{table_name}"

In [ ]:
class TestPythonCDCExpectations(NotebookTestFixture):
    """Tests for @dlt.expect_or_drop in python_cdc.ipynb"""
    
    def test_valid_records_pass(self, customers_cdc_schema, base_timestamp):
        data = [(1, "Alice", "Addr", "a@e.com", "APPEND", base_timestamp, None)]
        df = spark.createDataFrame(data, customers_cdc_schema)
        result = apply_dlt_expectations(df)
        assert result.count() == 1
    
    def test_null_id_dropped(self, customers_cdc_schema, base_timestamp):
        data = [(1, "A", "Addr", "a@e.com", "APPEND", base_timestamp, None),
                (None, "B", "Addr", "b@e.com", "APPEND", base_timestamp, None)]
        df = spark.createDataFrame(data, customers_cdc_schema)
        result = apply_dlt_expectations(df)
        assert result.count() == 1
    
    def test_invalid_operation_dropped(self, customers_cdc_schema, base_timestamp):
        data = [(1, "A", "Addr", "a@e.com", "INVALID", base_timestamp, None)]
        df = spark.createDataFrame(data, customers_cdc_schema)
        result = apply_dlt_expectations(df)
        assert result.count() == 0
    
    def test_rescued_data_dropped(self, customers_cdc_schema, base_timestamp):
        data = [(1, "A", "Addr", "a@e.com", "APPEND", base_timestamp, '{"x":1}')]
        df = spark.createDataFrame(data, customers_cdc_schema)
        result = apply_dlt_expectations(df)
        assert result.count() == 0

In [ ]:
class TestPythonCDCApplyChanges(NotebookTestFixture):
    """Tests for dlt.apply_changes in python_cdc.ipynb"""
    
    def test_append_inserts(self, customers_cdc_schema, customers_silver_schema, base_timestamp):
        src = spark.createDataFrame([(1, "Alice", "Addr", "a@e.com", "APPEND", base_timestamp, None)], customers_cdc_schema)
        src = apply_dlt_expectations(src)
        src = deduplicate_by_sequence(src)
        tgt = spark.createDataFrame([], customers_silver_schema)
        result = apply_changes_scd1(src, tgt)
        assert result.count() == 1
    
    def test_update_modifies(self, customers_cdc_schema, customers_silver_schema, base_timestamp):
        tgt = spark.createDataFrame([(1, "Alice Old", "Old", "old@e.com")], customers_silver_schema)
        src = spark.createDataFrame([(1, "Alice New", "New", "new@e.com", "UPDATE", base_timestamp, None)], customers_cdc_schema)
        src = apply_dlt_expectations(src)
        src = deduplicate_by_sequence(src)
        result = apply_changes_scd1(src, tgt)
        row = result.filter(col("id") == 1).collect()[0]
        assert row["name"] == "Alice New"
    
    def test_delete_removes(self, customers_cdc_schema, customers_silver_schema, base_timestamp):
        tgt = spark.createDataFrame([(1, "A", "Addr1", "a@e.com"), (2, "B", "Addr2", "b@e.com")], customers_silver_schema)
        src = spark.createDataFrame([(1, None, None, None, "DELETE", base_timestamp, None)], customers_cdc_schema)
        src = apply_dlt_expectations(src)
        src = deduplicate_by_sequence(src)
        result = apply_changes_scd1(src, tgt)
        assert result.count() == 1
        assert result.filter(col("id") == 2).count() == 1

In [ ]:
class TestPythonCDCDynamicPipeline(NotebookTestFixture):
    """Tests for dynamic pipeline creation in python_cdc.ipynb"""
    
    def test_folder_to_table_names(self):
        class F:
            def __init__(self, n): self.name = n
        folders = [F("customers/"), F("orders/"), F("products/")]
        names = get_table_names_from_folders(folders)
        assert names == ["customers", "orders", "products"]
    
    def test_raw_cdc_path(self):
        p = create_raw_cdc_path("customers", "my_catalog", "my_schema")
        assert p == "/Volumes/my_catalog/my_schema/raw_data/customers"
    
    def test_deduplication_keeps_latest(self, customers_cdc_schema, base_timestamp):
        data = [(1, "V1", "A1", "v1@e.com", "APPEND", base_timestamp, None),
                (1, "V2", "A2", "v2@e.com", "UPDATE", base_timestamp + timedelta(hours=1), None)]
        df = spark.createDataFrame(data, customers_cdc_schema)
        df = apply_dlt_expectations(df)
        result = deduplicate_by_sequence(df)
        assert result.count() == 1
        assert result.collect()[0]["name"] == "V2"

In [ ]:
run_notebook_tests()